# Sprint 4: Ensembles Predictivos y Comparación de Modelos

En este notebook entrenamos y evaluamos diferentes estrategias de ensamble (Voting, Bagging, Boosting y Stacking) utilizando los mejores modelos hiperparametrizados del Sprint 3 en validación cruzada. Comparamos sus métricas con el fin de seleccionar el mejor modelo para producción.

In [1]:
import sys
sys.path.append("../src")
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Cargar datos con variables sintéticas
df = pd.read_csv("../data/processed/features_data.csv")
X = df.drop(columns=['attrition'])
y = df['attrition']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")


X_train: (11200, 21), y_train: (11200,)


In [3]:
# Cargar preprocesador e hiperparámetros óptimos
preproc = joblib.load("../models/preprocessing_pipeline.pkl")

params_rf = joblib.load("../models/tuned_rf_optuna.pkl")
params_svm = joblib.load("../models/tuned_svm_optuna.pkl")
params_lr = joblib.load("../models/tuned_lr_optuna.pkl")
params_knn = joblib.load("../models/tuned_knn_optuna.pkl")

# Configurar lista de estimadores base
estimators_list = [
    ('rf', RandomForestClassifier(**params_rf)),
    ('svm', SVC(**params_svm)),
    ('lr', LogisticRegression(**params_lr)),
    ('knn', KNeighborsClassifier(**params_knn))
]


In [4]:
# Evaluar modelos base tuneados en validación cruzada (CV=5) con SMOTE
resultados_comparativos = {}
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'roc_auc']

for nombre, modelo in estimators_list:
    pipeline_baseline = ImbPipeline([
        ('preproc', preproc),
        ('smote', SMOTE(random_state=42)),
        ('clf', modelo)
    ])
    scores = cross_validate(pipeline_baseline, X_train, y_train, cv=5, scoring=scoring)
    resultados_comparativos[f"Optuna {nombre.upper()}"] = {m: scores[f'test_{m}'].mean() for m in scoring}
    print(f"Optuna {nombre.upper()} evaluado.")


Optuna RF evaluado.


Optuna SVM evaluado.


Optuna LR evaluado.


Optuna KNN evaluado.


In [5]:
# Hard Voting
voting_hard = ImbPipeline([
    ('preproc', preproc),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier(estimators_list, voting='hard'))
])
scoring_hard = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
scores_vh = cross_validate(voting_hard, X_train, y_train, cv=5, scoring=scoring_hard)
res_vh = {m: scores_vh[f'test_{m}'].mean() for m in scoring_hard}
res_vh['roc_auc'] = np.nan
resultados_comparativos["Ensemble Hard Voting"] = res_vh
print(f"Hard Voting F1: {res_vh['f1_macro']:.3f}")

# Soft Voting
voting_soft = ImbPipeline([
    ('preproc', preproc),
    ('smote', SMOTE(random_state=42)),
    ('voting', VotingClassifier(estimators_list, voting='soft'))
])
scores_vs = cross_validate(voting_soft, X_train, y_train, cv=5, scoring=scoring)
resultados_comparativos["Ensemble Soft Voting"] = {m: scores_vs[f'test_{m}'].mean() for m in scoring}
print(f"Soft Voting F1: {scores_vs['test_f1_macro'].mean():.3f}")


Hard Voting F1: 0.472


Soft Voting F1: nan


In [6]:
# Bagging Classifier sobre Random Forest
bagging = ImbPipeline([
    ('preproc', preproc),
    ('smote', SMOTE(random_state=42)),
    ('bagging', BaggingClassifier(estimator=RandomForestClassifier(**params_rf), n_estimators=10, random_state=42))
])
scores_bagging = cross_validate(bagging, X_train, y_train, cv=5, scoring=scoring)
resultados_comparativos["Ensemble Bagging (RF)"] = {m: scores_bagging[f'test_{m}'].mean() for m in scoring}
print(f"BaggingClassifier F1: {scores_bagging['test_f1_macro'].mean():.3f}")


BaggingClassifier F1: 0.496


In [7]:
# XGBoost
xgb_clf = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
xgb = ImbPipeline([('preproc', preproc), ('smote', SMOTE(random_state=42)), ('xgb', xgb_clf)])
scores_xgb = cross_validate(xgb, X_train, y_train, cv=5, scoring=scoring)
resultados_comparativos["Ensemble XGBoost"] = {m: scores_xgb[f'test_{m}'].mean() for m in scoring}
print(f"XGBoost F1: {scores_xgb['test_f1_macro'].mean():.3f}")

# LightGBM
lgbm_clf = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm = ImbPipeline([('preproc', preproc), ('smote', SMOTE(random_state=42)), ('lgbm', lgbm_clf)])
scores_lgbm = cross_validate(lgbm, X_train, y_train, cv=5, scoring=scoring)
resultados_comparativos["Ensemble LightGBM"] = {m: scores_lgbm[f'test_{m}'].mean() for m in scoring}
print(f"LightGBM F1: {scores_lgbm['test_f1_macro'].mean():.3f}")


XGBoost F1: 0.460


LightGBM F1: 0.458


In [8]:
# Stacking Classifier
stacking_clf = StackingClassifier(
    estimators=estimators_list, 
    final_estimator=LogisticRegression(), 
    cv=5
)
stacking = ImbPipeline([('preproc', preproc), ('smote', SMOTE(random_state=42)), ('stacking', stacking_clf)])
scores_stacking = cross_validate(stacking, X_train, y_train, cv=5, scoring=scoring)
resultados_comparativos["Ensemble Stacking"] = {m: scores_stacking[f'test_{m}'].mean() for m in scoring}
print(f"StackingClassifier F1: {scores_stacking['test_f1_macro'].mean():.3f}")


StackingClassifier F1: 0.502


In [9]:
# Consolidar y comparar métricas
df_resultados = pd.DataFrame(resultados_comparativos).T
df_resultados = df_resultados.sort_values(by='f1_macro', ascending=False)

def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: yellow' if v else '' for v in is_max]

styled_df = df_resultados.style.apply(highlight_max, subset=df_resultados.columns)
display(styled_df)


,accuracy,precision_macro,recall_macro,f1_macro,roc_auc
Ensemble Stacking,0.732946,0.502472,0.502880,0.502355,0.490067
Optuna RF,0.699821,0.500352,0.500559,0.497344,0.498446
Ensemble Bagging (RF),0.715357,0.497833,0.497907,0.496450,0.498601
Ensemble Hard Voting,0.616696,0.492973,0.488323,0.472007,nan
Ensemble XGBoost,0.844554,0.467860,0.500023,0.459552,0.485119
Ensemble LightGBM,0.844018,0.422726,0.498997,0.457706,0.494612
Optuna KNN,0.505446,0.495137,0.490680,0.431467,0.490451
Optuna LR,0.493839,0.496616,0.493506,0.426822,0.492563
Optuna SVM,0.432232,0.496294,0.493036,0.393245,0.491509
Ensemble Soft Voting,nan,nan,nan,nan,nan


In [10]:
# Entrenar y guardar el Stacking final como el modelo oficial
stacking.fit(X_train, y_train)
from pathlib import Path
Path("../models").mkdir(parents=True, exist_ok=True)
joblib.dump(stacking, "../models/final_model.pkl")
print("Guardado models/final_model.pkl con éxito.")


Guardado models/final_model.pkl con éxito.
